In [18]:
manual_q_override_raw = {
    "Cogent Economics and Finance": "Q2",
    "Cogent Business and Management": "Q2",
    "ISRA International Journal of Islamic Finance": "no-Q",
    "Journal of Asian Finance, Economics and Business": "no-Q",
    "Journal of Ecohumanism": "no-Q",
    "Journal of Population and Social Studies": "Q2",
    "Journal of Security and Sustainability Issues": "no-Q",
    "Review of International Geographical Education Online": "no-Q",
    "The Journal of Behavioral Science": "Q3",
    "Library Philosophy and Practice": "no-Q",
    "Systematic Reviews in Pharmacy": "no-Q",
    "Economics and Sociology": "Q2",
    "Economic Research-Ekonomska Istraživanja": "Q2",
    "Developmental Medicine and Child Neurology": "Q1",
    "Corporate Governance": "Q1",
    "Compensation and Benefits Review": "Q2",
    "Nonprofit Management and Leadership": "Q2",
    "Technology Analysis and Strategic Management": "Q2",
    "Leadership and Organization Development Journal": "Q1",
    "Social Sciences and Humanities Open": "Q1",
    "Journal of Accounting and Organizational Change": "Q1",
    "Journal of Nonprofit and Public Sector Marketing": "Q2",
    "Journal of Open Innovation: Technology, Market, and Complexity": "Q1",
    "International Journal of Financial Studies": "Q2",
    "International Journal of Productivity and Quality Management": "Q4",
    "International Journal of Trade and Global Markets": "Q4",
    "Journal of Sustainable Development of Energy, Water and Environment Systems": "Q2",
    "Corporate Social Responsibility and Environmental Management": "Q1",
    "Pertanika Journal of Social Sciences and Humanities": "Q2",
    "International Journal of Economics and Management": "Q3",
    "Journal of Management Development": "Q1",
    "Finance: Theory and Practice": "Q3",
    "Al-Ihkam: Jurnal Hukum dan Pranata Sosial": "Q1",
    "Vlakna a Textil": "Q4",
}

In [19]:
import re
import pandas as pd

COL_JOURNAL = "Journal"
COL_Q = "SCOPUS Q"

def override_key(value):
    if pd.isna(value):
        return ""
    
    text = str(value).strip().lower()
    text = re.sub(r"\s+", " ", text)

    # Samakan variasi umum
    text = text.replace("&", "and")
    text = text.replace("–", "-").replace("—", "-")

    # Hilangkan singkatan di akhir seperti (IJFS), (JPSS), [JPSS]
    text = re.sub(r"\s*\([a-z0-9\-]+\)\s*$", "", text)
    text = re.sub(r"\s*\[[a-z0-9\-]+\]\s*$", "", text)

    # Pertahankan hyphen dan colon
    text = re.sub(r"[^a-z0-9\s\-\:]", "", text)
    text = re.sub(r"\s*-\s*", "-", text)
    text = re.sub(r"\s*:\s*", ": ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [21]:
import pandas as pd
df_checkpoint2=pd.read_excel(r"D:\Proyek FEB\lengkapin data\sudah 40% pakai ini.xlsx")

In [22]:
manual_q_override = {
    override_key(journal): q
    for journal, q in manual_q_override_raw.items()
}

df_manual = df_checkpoint2.copy()  # atau ganti dengan nama dataframe checkpoint terakhirmu
df_manual["override_key"] = df_manual[COL_JOURNAL].apply(override_key)

override_count = 0

for idx, row in df_manual.iterrows():
    key = row["override_key"]
    
    if key in manual_q_override:
        new_q = manual_q_override[key]
        old_q = df_manual.at[idx, COL_Q]
        
        if old_q != new_q:
            df_manual.at[idx, COL_Q] = new_q
            override_count += 1
        
        if "q_source_checkpoint2" in df_manual.columns:
            df_manual.at[idx, "q_source_checkpoint2"] = "manual_verified_scimago_website"

print("Jumlah baris yang terkena manual override:", override_count)

Jumlah baris yang terkena manual override: 121


In [23]:
df_manual[COL_Q].value_counts(dropna=False)

SCOPUS Q
Q1      572
Q2      558
?       324
no-Q    311
Q3      253
Q4      135
Name: count, dtype: int64

In [24]:
df_manual[df_manual["override_key"].isin(manual_q_override.keys())][
    [COL_JOURNAL, COL_Q]
].drop_duplicates().sort_values(COL_JOURNAL)

,Journal,SCOPUS Q
2015,Al-Ihkam: Jurnal Hukum dan Pranata Sosial,Q1
2019,COGENT ECONOMICS & FINANCE,Q2
163,Cogent Business & Management,Q2
611,Cogent Business and Management,Q2
288,Cogent Economics & Finance,Q2
287,Cogent Economics and Finance,Q2
1932,Compensation & Benefits Review,Q2
799,Corporate Governance,Q1
882,Corporate Governance (Bingley),Q1
809,Corporate Social Responsibility and Environmen...,Q1


In [25]:
df_final = df_manual.drop(columns=["override_key"])

output_file = "45%.xlsx"
df_final.to_excel(output_file, index=False)

print(f"Saved: {output_file}")

Saved: 45%.xlsx


In [26]:
import pandas as pd
import re
from pathlib import Path

In [27]:
input_file = Path(r"D:\Proyek FEB\lengkapin data\45%.xlsx")

df = pd.read_excel(input_file)

COL_JOURNAL = "Journal"
COL_Q = "SCOPUS Q"

In [32]:
def is_empty_value(value):
    if pd.isna(value):
        return True
    
    text = str(value).strip()
    
    if text == "":
        return True
    
    if text.lower() in [
        "nan",
        "none",
        "null",
        "-",
        "?",
        "??",
        "???",
        "n/a",
        "na",
        "tidak ada",
        "not found"
    ]:
        return True
    
    return False


def journal_key(value):
    """
    Untuk internal fill, cukup samakan berdasarkan nama jurnal apa adanya:
    - lowercase
    - trim
    - rapikan spasi
    """
    if pd.isna(value):
        return ""
    
    text = str(value).strip().lower()
    text = re.sub(r"\s+", " ", text)
    
    return text

In [33]:
df_work = df.copy()

df_work["journal_key"] = df_work[COL_JOURNAL].apply(journal_key)
df_work["q_empty"] = df_work[COL_Q].apply(is_empty_value)

In [34]:
def first_non_empty(series):
    values = [
        str(v).strip()
        for v in series
        if not pd.isna(v) and str(v).strip() != ""
    ]
    
    return values[0] if values else ""


def join_filled_q(series):
    values = sorted(set(
        str(v).strip()
        for v in series
        if not is_empty_value(v)
    ))
    
    return " | ".join(values)


mixed_journals = (
    df_work.groupby("journal_key", dropna=False)
    .agg(
        rows=("journal_key", "size"),
        empty_q=("q_empty", "sum"),
        filled_q=("q_empty", lambda x: (~x).sum()),
        journal_name=(COL_JOURNAL, first_non_empty),
        q_values=(COL_Q, join_filled_q),
    )
    .reset_index()
)

mixed_journals = mixed_journals[
    (mixed_journals["empty_q"] > 0) &
    (mixed_journals["filled_q"] > 0)
].copy()

mixed_journals.sort_values("empty_q", ascending=False).head(50)

,journal_key,rows,empty_q,filled_q,journal_name,q_values
218,"international journal of innovation, creativit...",149,79,70,"International Journal of Innovation, Creativit...",no-Q
200,international journal of energy economics and ...,27,25,2,International Journal of Energy Economics and ...,Q1
376,opcion,39,21,18,Opcion,no-Q
61,buku,42,20,22,Buku,Q2 | Q4 | no-Q
242,international journal of supply chain management,20,16,4,International Journal of Supply Chain Management,no-Q
0,,9,8,1,,Q3
158,igi global,11,7,4,IGI Global,no-Q
12,- review of international geographical educati...,13,5,8,- Review of International Geographical Educati...,no-Q
159,ilkogretim online - elementary education online,6,5,1,Ilkogretim Online - Elementary Education Online,no-Q
62,buletin ekonomi moneter dan perbankan,7,4,3,Buletin Ekonomi Moneter dan Perbankan,Q2 | Q3


In [35]:
def parse_q_values(q_values):
    if pd.isna(q_values) or str(q_values).strip() == "":
        return []
    
    return [
        q.strip()
        for q in str(q_values).split("|")
        if q.strip()
    ]


mixed_journals["q_list"] = mixed_journals["q_values"].apply(parse_q_values)
mixed_journals["q_count"] = mixed_journals["q_list"].apply(len)

safe_internal = mixed_journals[mixed_journals["q_count"] == 1].copy()
conflict_internal = mixed_journals[mixed_journals["q_count"] > 1].copy()

print("Jurnal aman internal fill:", len(safe_internal))
print("Jurnal konflik:", len(conflict_internal))

Jurnal aman internal fill: 19
Jurnal konflik: 2


In [36]:
conflict_internal[
    ["journal_key", "rows", "empty_q", "filled_q", "journal_name", "q_values"]
].to_excel("internal_fill_conflicts_manual_check.xlsx", index=False)

conflict_internal[
    ["journal_key", "rows", "empty_q", "filled_q", "journal_name", "q_values"]
]

,journal_key,rows,empty_q,filled_q,journal_name,q_values
61,buku,42,20,22,Buku,Q2 | Q4 | no-Q
62,buletin ekonomi moneter dan perbankan,7,4,3,Buletin Ekonomi Moneter dan Perbankan,Q2 | Q3


In [37]:
safe_internal["fill_q"] = safe_internal["q_list"].apply(lambda x: x[0])

safe_internal_map = (
    safe_internal
    .set_index("journal_key")["fill_q"]
    .to_dict()
)

print("Jumlah jurnal aman untuk internal fill:", len(safe_internal_map))

Jumlah jurnal aman untuk internal fill: 19


In [38]:
df_internal = df_work.copy()

internal_fill_count = 0

for idx, row in df_internal.iterrows():
    key = row["journal_key"]
    
    if key not in safe_internal_map:
        continue
    
    if is_empty_value(row[COL_Q]):
        df_internal.at[idx, COL_Q] = safe_internal_map[key]
        internal_fill_count += 1

print("Jumlah baris Q terisi dari internal fill aman:", internal_fill_count)

Jumlah baris Q terisi dari internal fill aman: 189


In [39]:
manual_internal_override = {
    "buletin ekonomi moneter dan perbankan": "Q3",
    "buku": "?",
}

manual_count = 0

for idx, row in df_internal.iterrows():
    key = row["journal_key"]
    
    if key in manual_internal_override:
        new_q = manual_internal_override[key]
        
        if df_internal.at[idx, COL_Q] != new_q:
            df_internal.at[idx, COL_Q] = new_q
            manual_count += 1

print("Jumlah baris Q diseragamkan dari manual override konflik:", manual_count)

Jumlah baris Q diseragamkan dari manual override konflik: 27


In [40]:
df_internal[
    df_internal["journal_key"].isin([
        "buletin ekonomi moneter dan perbankan",
        "buku"
    ])
][[COL_JOURNAL, COL_Q]].drop_duplicates().sort_values(COL_JOURNAL)

,Journal,SCOPUS Q
590,Buku,?
383,Buletin Ekonomi Moneter dan Perbankan,Q3


In [41]:
df_internal["q_empty"] = df_internal[COL_Q].apply(is_empty_value)

mixed_after = (
    df_internal.groupby("journal_key", dropna=False)
    .agg(
        rows=("journal_key", "size"),
        empty_q=("q_empty", "sum"),
        filled_q=("q_empty", lambda x: (~x).sum()),
        journal_name=(COL_JOURNAL, first_non_empty),
        q_values=(COL_Q, join_filled_q),
    )
    .reset_index()
)

mixed_after = mixed_after[
    (mixed_after["empty_q"] > 0) &
    (mixed_after["filled_q"] > 0)
].copy()

mixed_after.sort_values("empty_q", ascending=False)

,journal_key,rows,empty_q,filled_q,journal_name,q_values


In [42]:
df_internal[COL_Q].value_counts(dropna=False)

SCOPUS Q
Q1      600
Q2      561
no-Q    447
Q3      268
?       153
Q4      124
Name: count, dtype: int64

In [43]:
remaining_missing = df_internal[df_internal[COL_Q].apply(is_empty_value)].copy()

print("Sisa baris Q masih ?:", len(remaining_missing))

remaining_unique = (
    remaining_missing[[COL_JOURNAL, "journal_key"]]
    .drop_duplicates()
    .sort_values(COL_JOURNAL)
)

print("Sisa jurnal unik masih ?:", len(remaining_unique))

remaining_unique.to_excel("remaining_q_missing_after_internal_fill.xlsx", index=False)

remaining_unique.head(50)

Sisa baris Q masih ?: 153
Sisa jurnal unik masih ?: 50


,Journal,journal_key
641,- Ilkogretim Online - Elementary Education Onl...,- ilkogretim online - elementary education onl...
633,"- International Journal of Innovation, Creativ...","- international journal of innovation, creativ..."
1502,- Journal of Islamic Marketing\n- Studies in S...,- journal of islamic marketing - studies in sy...
640,- Review of International Geographical Educati...,- review of international geographical educati...
572,--,--
1840,2024 International Conference on Sustainable I...,2024 international conference on sustainable i...
1841,2025 International Conference on Sustainable I...,2025 international conference on sustainable i...
1062,AL-IHKAM Jurnal Hukum & Pranata Sosial,al-ihkam jurnal hukum & pranata sosial
618,Academy of Accounting and Financial Studies Jo...,academy of accounting and financial studies jo...
191,Academy of Strategic Management Journal,academy of strategic management journal


In [44]:
df_final = df_internal.drop(columns=["journal_key", "q_empty"], errors="ignore")

output_file = "46%.xlsx"

df_final.to_excel(output_file, index=False)

print(f"Saved: {output_file}")

Saved: 46%.xlsx
